Installing Required Libraries:

In [1]:
pip install -qU langchain-ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
pip install -qU langchain

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pip


     ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
     -- ------------------------------------- 0.1/1.8 MB 2.6 MB/s eta 0:00:01
     ------------- -------------------------- 0.6/1.8 MB 6.4 MB/s eta 0:00:01
     ------------------ --------------------- 0.8/1.8 MB 6.6 MB/s eta 0:00:01
     ------------------------------ --------- 1.4/1.8 MB 8.0 MB/s eta 0:00:01
     ---------------------------------------- 1.8/1.8 MB 7.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
Note: you may need to restart the kernel to use updated packages.


Model Initialization and basic setup:

In [4]:
!pip install ollama


In [13]:
import ollama

print("Downloading qwen2.5:3b... Please wait.")
ollama.pull('qwen2.5:3b')
print("Success! Model is ready to use.")


Success! Model is ready to use.


In [14]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0,
    # other params...
)



In [11]:
ai_msg.content

'2 + 2 = 4\n'

Defining Tools (Custom):

In [15]:
from langchain_core.tools import tool

In [16]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add two numbers together. Use this when the user wants to add or find the sum of two numbers."""
    return x + y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract the second number from the first. Use this when the user wants to subtract or find the difference between two numbers."""
    return x - y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together. Use this when the user wants to multiply or find the product of two numbers."""
    return x * y

@tool
def divide(x: float, y: float) -> float:
    """Divide the first number by the second. Use this when the user wants to divide or find the quotient of two numbers. Will return an error if dividing by zero."""
    if y == 0:
        raise ValueError("Cannot divide by zero.")
    return x / y

# Group all tools into a list — this is what gets passed to the agent later
tools = [add, subtract, multiply, divide]

Creating Prompt Template:

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a specialized AI math assistant. "
            "You help users solve math problems by using the available tools. "
            "Always use a tool to compute the answer. Never guess or calculate yourself."
        ),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

Creating Agent:

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    llm,                    # our ChatOllama model from Step 2
    tools=tools,            # our [add, subtract, multiply, divide] list from Step 3
    system_prompt=(
        "You are a specialized AI math assistant. "
        "Help users solve math problems by using the available tools. "
        "Always use a tool to compute the answer. Never guess or calculate yourself."
    )
)

In [19]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 25 multiplied by 4?"}]}
)

print(result["messages"][-1].content)

25 multiplied by 4 is 100.


In [ ]:
FINAL FLOW:

User Input
    ↓
create_agent  ←  system_prompt tells agent who it is
    ↓
ChatOllama    ←  reads input, decides which tool to call
    ↓
@tool         ←  actual Python math function runs
    ↓
Result        ←  agent reads it, loops or gives final answer
    ↓
result["messages"][-1].content  ←  your final output